## Set current dir to project root dir

In [1]:
path <- getwd()
markers <- c(".git", "Makefile", "renv.lock", ".Rprofile")
while (!any(file.exists(file.path(path, markers)))) {
  parent <- dirname(path)
  if (parent == path) stop("Could not find project root")
  path <- parent
}
setwd(path)
rm(path, parent, markers)

## Activate project's local R env. using renv

In [2]:
if (file.exists("renv/activate.R")) {
  source("renv/activate.R")
} else {
  stop("Could not find renv/activate.R in project root")
}

## Imports

In [3]:
library(readr)
library(eegUtils)


Attaching package: ‘eegUtils’


The following object is masked from ‘package:stats’:

    filter




## Helper functions

## `format_elapsed(seconds)`

Formats an elapsed time in seconds into a human-readable string following the
[NIST Guide to the SI, Chapter 7](https://www.nist.gov/pml/special-publication-811/nist-guide-si-chapter-7-rules-and-style-conventions-expressing-values)
conventions for expressing values of quantities with units.

Examples: `41.3 s`, `1 min 41.3 s`, `1 h 20 min 10.6 s`

In [4]:
format_elapsed <- function(seconds) {
  if (seconds >= 3600) {
    h <- floor(seconds / 3600)
    m <- floor((seconds %% 3600) / 60)
    s <- seconds %% 60
    sprintf("%d h %d min %.1f s", h, m, s)
  } else if (seconds >= 60) {
    m <- floor(seconds / 60)
    s <- seconds %% 60
    sprintf("%d min %.1f s", m, s)
  } else {
    sprintf("%.1f s", seconds)
  }
}

## Functions

In [5]:
get_subject_folders <- function(parent_directory) {
  folders <- list.dirs(parent_directory, full.names = TRUE, recursive = FALSE)
  folders <- folders[grepl("/sub-", folders)]
  sort(folders)
}

In [6]:
extract_unique_stimuli <- function(file_path) {
  # Parses a BrainVision .vmrk file and returns a sorted list
  # of all unique stimulus descriptions (trigger codes).
  stimuli <- c()
  lines <- readLines(file_path, encoding = "UTF-8")
  for (line in lines) {
    if (grepl("^Mk", line)) {
      tryCatch({
        content <- strsplit(line, "=")[[1]][2]
        parts <- strsplit(content, ",")[[1]]
        marker_type <- trimws(parts[1])
        description <- trimws(parts[2])
        if (marker_type == "Stimulus") {
          stimuli <- c(stimuli, description)
        }
      }, error = function(e) NULL)
    }
  }
  sort(unique(stimuli))
}

## Variables

In [7]:
main_data_folder <- "./ds006018"
tasks <- c("task-auditoryoddball", "task-flanker",
           "task-visualoddball", "task-visualsearch")

## Main

In [8]:
# ── Timer start ───────────────────────────────────────────────────────────────────────
start_time <- proc.time()
# ──────────────────────────────────────────────────────────────────────────────────────

library(eegUtils)
library(readr)

# ── Helpers ───────────────────────────────────────────────────────────────────

get_subject_folders <- function(main_folder) {
  dirs <- list.dirs(main_folder, full.names = TRUE, recursive = FALSE)
  dirs[!grepl("^\\..*", basename(dirs))]  # exclude hidden folders like .datalad
}

extract_unique_stimuli <- function(path_to_vmrk) {
  lines          <- readLines(path_to_vmrk)
  stimulus_lines <- lines[grepl("^Mk.*=Stimulus", lines)]
  stimuli        <- sub(".*=Stimulus,([^,]+),.*", "\\1", stimulus_lines)
  unique(trimws(stimuli))
}

# ── Standard 10-20 electrode locations ───────────────────────────────────────
# eegUtils does not support "standard_1020" as a montage string.
# Coordinates below match MNE's standard_1020 montage for the 27 scalp
# channels present in this dataset. Non-EEG channels (HEL, HER, VER, LM, RM)
# are assigned NA and are harmless.

locs <- data.frame(
  electrode = c("Fp1","Fp2","Fz","F3","F4","F7","F8",
                "FC1","FC2","FC5","FC6",
                "Cz","C3","C4","T7","T8",
                "CP1","CP2","CP5","CP6",
                "Pz","P3","P4","P7","P8",
                "O1","O2","Oz"),
  x = c(-0.308, 0.308, 0.000,-0.231, 0.231,-0.587, 0.587,
         -0.197, 0.197,-0.573, 0.573,
          0.000,-0.397, 0.397,-0.719, 0.719,
         -0.197, 0.197,-0.573, 0.573,
          0.000,-0.231, 0.231,-0.587, 0.587,
         -0.308, 0.308, 0.000),
  y = c( 0.756, 0.756, 0.656, 0.594, 0.594, 0.381, 0.381,
          0.370, 0.370, 0.095, 0.095,
          0.000, 0.000, 0.000, 0.000, 0.000,
         -0.370,-0.370,-0.095,-0.095,
         -0.656,-0.594,-0.594,-0.381,-0.381,
         -0.756,-0.756,-0.900),
  stringsAsFactors = FALSE
)

assign_montage <- function(raw) {
  matched        <- locs[match(channel_names(raw), locs$electrode), ]
  channels(raw)  <- matched
  raw
}

# ── Main loop ─────────────────────────────────────────────────────────────────

subject_folders <- get_subject_folders(main_data_folder)
n_subjects      <- length(subject_folders)

for (i in seq_along(subject_folders)) {
  subject_path <- subject_folders[[i]]
  sub_number   <- basename(subject_path)

  cat(sprintf("[%d/%d] Processing subject: %s\n", i, n_subjects, sub_number))

  out_dir <- file.path("./ds006018_per_stimuli", sub_number)
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

  for (task in tasks) {

    path_to_vhdr <- file.path(subject_path, "eeg",
                              paste0(sub_number, "_", task, "_eeg.vhdr"))
    path_to_vmrk <- file.path(subject_path, "eeg",
                              paste0(sub_number, "_", task, "_eeg.vmrk"))

    if (!file.exists(path_to_vhdr)) {
      cat(path_to_vhdr, "File not found.\n")
      next
    }

    cat("The file exists!\n")

    # Get unique stimulus codes from .vmrk
    unique_stimuli <- extract_unique_stimuli(path_to_vmrk)
    cat("Stimuli found:", paste(unique_stimuli, collapse = ", "), "\n")

    # 1. Load BrainVision data
    raw <- import_raw(path_to_vhdr)

    # 2. Set standard 10-20 montage
    raw <- assign_montage(raw)

    # 3. Bandpass filter 0.1–40 Hz
    raw <- eeg_filter(raw, low_freq = 0.1, high_freq = 40.0, method = "iir")

    # 4. Epoch per stimulus, baseline correct, export
    for (stimulus in unique_stimuli) {
      clean_name <- gsub("[/ ]", "", stimulus)
      clean_name <- paste0("Stimulus_", clean_name)

      tryCatch({
        # time_lim specifies the epoch window in seconds relative to the event.
        # epoch_start/epoch_end are not valid eegUtils parameters — use time_lim instead.
        epochs <- epoch_data(raw,
                             events   = stimulus,
                             time_lim = c(-0.2, 0.8))

        n_epochs <- length(unique(epochs$timings$epoch))

        if (n_epochs == 0) {
          cat(sprintf("Skipping %s: 0 trials found\n", stimulus))
          next
        }

        # 5. Baseline correction [-0.2, 0.0]
        epochs <- rm_baseline(epochs, baseline = c(-0.2, 0))

        # 6. Export to CSV
        # unique() removes duplicate rows produced by eegUtils's as.data.frame()
        # due to an internal join between the signals and timings tables.
        df           <- as.data.frame(epochs)
        df           <- unique(df)
        channel_cols <- setdiff(colnames(df), c("time", "epoch", "recording",
                                                 "event_type", "participant_id",
                                                 "condition", "epoch_label", "sample"))
        df           <- df[, c("time", "epoch", channel_cols)]
        out_path     <- file.path(out_dir, paste0(task, "_", clean_name, ".csv"))
        write_csv(df, out_path)

        cat(sprintf("Successfully created: eeg_%s.csv (%d trials)\n",
                    clean_name, n_epochs))

      }, error = function(e) {
        cat(sprintf("Skipping %s: %s\n", stimulus, conditionMessage(e)))
      })
    }
  }
}

# ── Timer end ─────────────────────────────────────────────────────────────────────────
cat("───────────────────────────────────────────────────────────────────────────────\n")
cat(sprintf("  Time elapsed: %s\n",format_elapsed((proc.time()-start_time)["elapsed"])))
cat("───────────────────────────────────────────────────────────────────────────────\n")
# ──────────────────────────────────────────────────────────────────────────────────────

[1/127] Processing subject: sub-001
The file exists!
Stimuli found: S  1, S180, S 80, S 70 


Importing Brain Vision Analyzer file ./ds006018/sub-001/eeg/sub-001_task-auditoryoddball_eeg.vhdr

Band-pass IIR filter from 0.1 - 40 Hz

Effective filter order: 4 (two-pass)

Removing channel means...

Output limits:  -0.2 0.8

No baseline removal performed.

Creating 1 epochs.

Removing channel means per epoch...



Successfully created: eeg_Stimulus_S1.csv (1 trials)


Output limits:  -0.2 0.8

No baseline removal performed.

Creating 15 epochs.

Removing channel means per epoch...



Successfully created: eeg_Stimulus_S180.csv (15 trials)


...

Successfully created: eeg_Stimulus_S53.csv (8 trials)


Output limits:  -0.2 0.8

No baseline removal performed.

Creating 8 epochs.

Removing channel means per epoch...



Successfully created: eeg_Stimulus_S55.csv (8 trials)


Output limits:  -0.2 0.8

No baseline removal performed.

Creating 8 epochs.

Removing channel means per epoch...



Successfully created: eeg_Stimulus_S51.csv (8 trials)
./ds006018/sub-127/eeg/sub-127_task-visualsearch_eeg.vhdr File not found.
───────────────────────────────────────────────────────────────────────────────
  Time elapsed: 44 min 6.5 s
───────────────────────────────────────────────────────────────────────────────
